# Sentinel-2: search and download L1C products

The Sentinel-2 L1C products are distributed by the [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu)
(CDSE). This tutorial searches the CDSE catalogue for the products of a tile and a period, and
downloads them into a local archive organized as `<tile>/<year>/<month>/<day>/*.SAFE`, the layout
used in the [subsetting tutorial](GRSdriver_subsetting.ipynb).

**You will learn to**

- query the CDSE catalogue (OData API, no account needed),
- keep one product per acquisition (the latest processing baseline),
- download the products with `s3cmd` or with HTTPS.

**Requirements**: a free CDSE account for the download only, `requests`, and `s3cmd` for the S3 download.

In [1]:
import os
from datetime import datetime
from multiprocessing import Pool
import subprocess

import pandas as pd
import requests

## Search the catalogue

The [OData API](https://documentation.dataspace.copernicus.eu/APIs/OData.html) of the catalogue
takes a filter on the collection, the product type, the name (here the tile), the cloud cover
and the acquisition date:

In [2]:
tiles = ['31TFJ']
date_start, date_end = '2022-07-01T00:00:00.000Z', '2022-08-31T00:00:00.000Z'
cloud_cover_max = 10   # % of the tile

catalogue = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'

def search(tile):
    query = ("Collection/Name eq 'SENTINEL-2'"
             " and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType'"
             " and att/OData.CSC.StringAttribute/Value eq 'S2MSI1C')"
             " and Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover'"
             f" and att/OData.CSC.DoubleAttribute/Value lt {cloud_cover_max:.2f})"
             f" and contains(Name,'T{tile}')"
             f" and ContentDate/Start gt {date_start} and ContentDate/Start lt {date_end}")
    response = requests.get(catalogue, params={'$filter': query, '$orderby': 'ContentDate/Start',
                                               '$top': 1000}, timeout=60)
    response.raise_for_status()
    return pd.DataFrame(response.json()['value'])

products = pd.concat([search(tile) for tile in tiles], ignore_index=True)
print(len(products), 'products found')
products[['Name', 'ContentLength', 'S3Path']].head()

22 products found


,Name,ContentLength,S3Path
0,S2B_MSIL1C_20220701T103629_N0510_R008_T31TFJ_2...,774383079,/eodata/Sentinel-2/MSI/L1C_N0500/2022/07/01/S2...
1,S2A_MSIL1C_20220703T102611_N0510_R108_T31TFJ_2...,839967236,/eodata/Sentinel-2/MSI/L1C_N0500/2022/07/03/S2...
2,S2A_MSIL1C_20220706T103641_N0510_R008_T31TFJ_2...,770948513,/eodata/Sentinel-2/MSI/L1C_N0500/2022/07/06/S2...
3,S2B_MSIL1C_20220708T102559_N0510_R108_T31TFJ_2...,855714814,/eodata/Sentinel-2/MSI/L1C_N0500/2022/07/08/S2...
4,S2B_MSIL1C_20220711T103629_N0510_R008_T31TFJ_2...,769681118,/eodata/Sentinel-2/MSI/L1C_N0500/2022/07/11/S2...


## One product per acquisition

The archive can hold several versions of the same acquisition, reprocessed with different
processing baselines (`N0400`, `N0510`...). The product name gives the acquisition time, the
baseline and the tile: keep the latest baseline of each acquisition.

In [3]:
name = products.Name.str.replace('.SAFE', '').str.split('_', expand=True)
products['date'] = pd.to_datetime(name[2], format='%Y%m%dT%H%M%S')
products['baseline'] = name[3]
products['tile'] = name[5].str[1:]
products['size_MB'] = (products.ContentLength / 1e6).round()

products = (products.sort_values('baseline')
            .drop_duplicates(subset=['date', 'tile'], keep='last')
            .sort_values('date').reset_index(drop=True))
print(len(products), 'acquisitions')
products[['date', 'tile', 'baseline', 'size_MB', 'Name']]

21 acquisitions


,date,tile,baseline,size_MB,Name
0,2022-07-01 10:36:29,31TFJ,N0510,774.0,S2B_MSIL1C_20220701T103629_N0510_R008_T31TFJ_2...
1,2022-07-03 10:26:11,31TFJ,N0510,840.0,S2A_MSIL1C_20220703T102611_N0510_R108_T31TFJ_2...
2,2022-07-06 10:36:41,31TFJ,N0510,771.0,S2A_MSIL1C_20220706T103641_N0510_R008_T31TFJ_2...
3,2022-07-08 10:25:59,31TFJ,N0510,856.0,S2B_MSIL1C_20220708T102559_N0510_R108_T31TFJ_2...
4,2022-07-11 10:36:29,31TFJ,N0510,770.0,S2B_MSIL1C_20220711T103629_N0510_R008_T31TFJ_2...
5,2022-07-13 10:30:41,31TFJ,N0510,853.0,S2A_MSIL1C_20220713T103041_N0510_R108_T31TFJ_2...
6,2022-07-16 10:36:41,31TFJ,N0510,773.0,S2A_MSIL1C_20220716T103641_N0510_R008_T31TFJ_2...
7,2022-07-18 10:25:59,31TFJ,N0511,832.0,S2B_MSIL1C_20220718T102559_N0511_R108_T31TFJ_2...
8,2022-07-21 10:36:29,31TFJ,N0510,541.0,S2B_MSIL1C_20220721T103629_N0510_R008_T31TFJ_2...
9,2022-07-26 10:36:31,31TFJ,N0510,781.0,S2A_MSIL1C_20220726T103631_N0510_R008_T31TFJ_2...


## Download with s3cmd

The S3 interface is the fastest way to download many products. Create S3 credentials in the
[CDSE dashboard](https://eodata-s3keysmanager.dataspace.copernicus.eu) and write them in an `s3cmd`
configuration file (here `~/.s3cfg`), as explained in the
[CDSE documentation](https://documentation.dataspace.copernicus.eu/APIs/S3.html).

One command per product, to download the `.SAFE` directory into the archive:

In [4]:
archive = '/data/satellite/Sentinel-2/L1C'

def s3cmd_command(product):
    odir = os.path.join(archive, product.tile, product.date.strftime('%Y/%m/%d'))
    return f's3cmd -c ~/.s3cfg sync s3:/{product.S3Path} {odir}/'

commands = [s3cmd_command(product) for product in products.itertuples()]
print(*commands[:3], '...', sep='\n')

s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2022/07/01/S2B_MSIL1C_20220701T103629_N0510_R008_T31TFJ_20240702T081916.SAFE /data/satellite/Sentinel-2/L1C/31TFJ/2022/07/01/
s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2022/07/03/S2A_MSIL1C_20220703T102611_N0510_R108_T31TFJ_20240701T012044.SAFE /data/satellite/Sentinel-2/L1C/31TFJ/2022/07/03/
s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2022/07/06/S2A_MSIL1C_20220706T103641_N0510_R008_T31TFJ_20240630T200735.SAFE /data/satellite/Sentinel-2/L1C/31TFJ/2022/07/06/
...


Run the commands in parallel (set `download = True`; already downloaded files are skipped by `sync`):

In [5]:
download = False

def run(command):
    return subprocess.call(command, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

if download:
    with Pool(processes=4) as pool:
        status = pool.map(run, commands)
    print(f'{status.count(0)} / {len(commands)} downloads succeeded')

## Download with HTTPS

Without `s3cmd`, a product can be downloaded as a zip file with an access token of your CDSE
account. Set your credentials in the environment variables `CDSE_USER` and `CDSE_PASSWORD` (never
write them in a notebook):

In [6]:
def access_token(user, password):
    response = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={'client_id': 'cdse-public', 'grant_type': 'password', 'username': user, 'password': password},
        timeout=60)
    response.raise_for_status()
    return response.json()['access_token']

def download_zip(product, token, odir='.'):
    url = f'https://zipper.dataspace.copernicus.eu/odata/v1/Products({product.Id})/$value'
    file = os.path.join(odir, product.Name.replace('.SAFE', '.zip'))
    with requests.Session() as session:
        session.headers.update({'Authorization': f'Bearer {token}'})
        with session.get(url, stream=True, timeout=600) as response:
            response.raise_for_status()
            with open(file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
    return file

if download:
    token = access_token(os.environ['CDSE_USER'], os.environ['CDSE_PASSWORD'])
    print(download_zip(products.iloc[0], token))

The zip file contains the `.SAFE` directory: unzip it in the archive. The access token is valid
for 10 minutes: request a new one for long downloads.

## Next steps

- [Load an image](GRSdriver_basics.ipynb) or [subset a series](GRSdriver_subsetting.ipynb) of the downloaded products.